<center><p float="center">
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<h1><center> Real-Time Retail Feedback Intelligence
 </center></h1>


### **Business Context**
Why is this problem important to solve?

ChicStyle is a growing fashion retail platform that sees massive spikes in customer activity during festive seasons and holiday sales. As shoppers buy clothing and accessories for celebrations, the volume of incoming reviews rises sharply — they arrive every hour, ranging from positive praise to urgent complaints about fit, delivery delays, product defects, or sizing issues. During these high-pressure periods, even a slight delay in reading or responding to feedback can have serious consequences. If the retail team fails to act quickly, customers feel ignored at an emotionally significant time, which leads to frustration, spoiled shopping experiences, and reduced trust in the brand. The cost is twofold: immediate lost sales, and longer-term damage to repeat purchases and loyalty. To protect both revenue and reputation at peak periods, ChicStyle needs a system that can process thousands of reviews instantly, accurately, and with business context.

Customer reviews are direct, unsolicited signals about what is working and what is failing across the product and service experience. Analyzing them well lets a retailer:

- **Catch problems early** — defects, sizing inconsistencies, and delivery failures surface in reviews before they show up in returns and churn.
- **Prioritize action** — separating urgent complaints from minor suggestions lets limited support and operations capacity focus where it matters most.
- **Improve products faster** — recurring themes (e.g., a dress that runs small) feed directly back to merchandising and quality teams.
- **Protect brand reputation and loyalty** — responding promptly, especially to negative feedback, signals that the brand listens and converts a bad experience into a recoverable one.
- **Turn unstructured text into business intelligence** — at scale, reviews become a measurable, trackable input to decisions rather than anecdotes.

The challenge is volume and nuance. Traditional rule-based NLP struggles with mixed feedback — for example, *"The fit is great but the color was not as per the product image"* contains both a positive and a negative opinion, and older systems tend to collapse it into a single sentiment, losing critical detail. This is precisely the gap a Generative AI approach is meant to close.


### **Objective**
What is the intended goal?

Build a **Generative AI feedback system** that uses prompt engineering — **Zero-Shot, Few-Shot, and Chain-of-Thought (CoT)** prompting — to transform large volumes of raw customer reviews into structured, actionable intelligence. Specifically, the system should:

- **Analyze and categorize sentiment in real time** (positive, negative, neutral), including mixed feedback.
- **Detect which product or service** each piece of feedback refers to (e.g., fit, quality, delivery, price), even though there is no explicit service column — it is inferred from the review text.
- **Summarize insights by product category and urgency level**, where category comes from existing department data and urgency (high / medium / low) is inferred from the language of the review.
- **Automatically send short, personalized messages** to customers based on sentiment — thanking them for positive feedback, acknowledging neutral comments, and apologizing for negative ones while letting them know a team member will follow up.
- **Generate short, actionable reports** for retail teams.

Manual review reading does not scale to thousands of reviews per hour during peak sales, and it is slow, inconsistent, and expensive. Automating it lets ChicStyle:

- **Respond in real time** rather than hours or days later, so customers feel heard during emotionally important purchases.
- **Act on issues quickly** — high-urgency complaints are flagged and routed instead of being buried in a backlog.
- **Improve product quality faster** by surfacing recurring themes automatically.
- **Apply consistent, context-aware judgment** at scale, capturing nuance that rule-based models miss.
- **Free up human teams** to focus on resolution and relationships instead of triage.

In short, the goal is to turn massive unstructured feedback into meaningful, real-time business intelligence that protects revenue, reputation, and loyalty during the moments that matter most.

### **Dataset Used for the Notebook**
Describe dataset used for this project.

#### **Description**

The project uses the **"Women's E-Commerce Clothing Reviews"** dataset: **23,486 reviews with 10 columns**. 
Each row is a single customer review of a clothing item, combining structured metadata (rating, recommendation, department) with unstructured review text.

| Column                  | Description                                                  | Type             |
| ----------------------- | ------------------------------------------------------------ | ---------------- |
| Clothing.ID             | Unique ID for each piece of clothing                         | Integer          |
| Age                     | Age of the reviewer                                          | Positive integer |
| Title                   | Title of the review                                          | String           |
| Review.Text             | Main body of the review                                      | String           |
| Rating                  | Product score, 1 (worst) to 5 (best)                         | Ordinal integer  |
| Recommended.IND         | Whether the customer recommends the product (1 = yes, 0 = no) | Binary           |
| Positive.Feedback.Count | Number of other customers who found the review helpful       | Positive integer |
| Division.Name           | High-level division of the product                           | Categorical      |
| Department.Name         | Specific department of the product                           | Categorical      |
| Class.Name              | Product class (finer than department)                        | Categorical      |

Reviewers skew middle-aged: ages range from **18 to 99** with a **mean of ~43** and a median of 41. Reviews average about **60 words** each.

> **Note on scope:** For the GenAI prompt-engineering and evaluation work, only a **sample of 50 reviews** is processed (5–10 during initial prompt testing, scaling to 50 for final evaluation) to stay within the API budget. The full 23,486-row dataset is only used for the exploratory analysis.

#### **Key Patterns Summary**
*Ratings — strongly skewed positive:*

| Rating | Count  | Share |
| ------ | ------ | ----- |
| 5      | 13,131 | 55.9% |
| 4      | 5,077  | 21.6% |
| 3      | 2,871  | 12.2% |
| 2      | 1,565  | 6.7%  |
| 1      | 842    | 3.6%  |

About **77.5% of reviews are 4–5 stars** and **82.2% of customers recommend** the product. This class imbalance matters: negative reviews are the minority but carry the highest business urgency, so the system must not let the volume of praise drown out the smaller stream of complaints. Rating aligns tightly with recommendation — the average rating is **4.6 among recommenders vs. 2.3 among non-recommenders** — confirming the two signals are consistent.

*Departments — concentrated in a few categories:*

| Department | Count  |
| ---------- | ------ |
| Tops       | 10,468 |
| Dresses    | 6,319  |
| Bottoms    | 3,799  |
| Intimate   | 1,735  |
| Jackets    | 1,032  |
| Trend      | 119    |

**Tops and Dresses together account for ~71%** of all reviews, so feedback volume — and therefore the operational load and the biggest opportunity for product improvement — is concentrated there. At the division level, **General (13,850)** and **General Petite (8,120)** dominate, with Intimates smallest. The finest level (Class.Name) is led by Dresses, Knits, and Blouses.

*Fit and sizing dominate both ends:*

- **Positive reviews (4–5★):** *love, great, perfect, fit, size, fabric, color, dress, top, comfortable* — praise centers on fit, fabric, and overall delight.
- **Negative reviews (1–2★):** *too, small, fabric, fit, size, ordered, look, back* — complaints cluster around **fit/sizing** ("too small"), **fabric quality**, and **returns** ("ordered… back").

The recurring takeaway is that **fit and sizing are the single most important theme across the entire experience** — they drive both the strongest praise and the strongest complaints. Fabric/quality and color-vs-image expectations are the next most common pain points.

#### **Data treatments / preprocessing required**
- **Missing values:** `Review.Text` is missing in **845 rows** and `Title` in **3,810 rows**; `Division.Name`, `Department.Name`, and `Class.Name` are each missing in **14 rows**. Rows with no review text cannot be analyzed by the language model and should be dropped (or excluded from the sampled set); the 14 rows missing category fields can be dropped or imputed as "Unknown."
- **CSV parsing:** the file is **semicolon-delimited** with an unnamed leading index column — it must be read with the correct separator and index handling, and review text contains embedded quotes/commas that must be parsed cleanly.
- **Text cleaning for analysis:** for EDA and word clouds, lowercase the text, strip punctuation, and remove stopwords; keep the raw text intact for the LLM so it retains nuance.
- **Sampling:** draw a representative ~50-review sample for prompt experiments, ideally spanning the rating range and major departments so evaluation isn't dominated by 5-star Tops reviews.
- **Type checks:** confirm `Rating` (1–5) and `Recommended.IND` (0/1) fall in valid ranges; treat `Rating`, `Department.Name`, and `Division.Name` as categ/ordinal as appropriate.
- **No new labels needed for category/urgency:** the project intentionally does **not** add sentiment, "Category," or "urgency" columns to the data — these are generated by the LLM from the review text at analysis time, with product category drawn from the existing `Department.Name`.

### **Installing and Importing Necessary Libraries**
First, let's set up the environment by installing the required Python libraries.

In [1]:
# Install the required libraries for the project
# We use `uv` (a fast Python package manager) instead of pip.
# The `!` prefix runs these as shell commands from within the notebook.
# `uv add` adds each package to the project's environment (pyproject.toml / uv.lock).

# --- Data handling & numerical computing ---
!uv add numpy pandas

# --- Visualization libraries ---
!uv add matplotlib seaborn wordcloud plotly

# --- Machine learning metrics (scikit-learn) ---
!uv add scikit-learn

# --- Progress bars for long-running loops ---
!uv add tqdm

# --- OpenAI client for LLM-based analysis ---
!uv add openai

# --- IPython for rich in-notebook display (display, etc.) ---
!uv add ipython


Resolved 68 packages in 3ms
Audited 63 packages in 4ms
Resolved 68 packages in 3ms
Audited 63 packages in 3ms
Resolved 68 packages in 3ms
Audited 63 packages in 3ms
Resolved 68 packages in 3ms
Audited 63 packages in 3ms
Resolved 68 packages in 3ms
Audited 63 packages in 3ms
Resolved 68 packages in 3ms
Audited 63 packages in 3ms


In [2]:
# Import the required libraries for the project

# --- Core Utilities ---
import time
import json
import re
import numpy as np
from typing import Dict, List, Optional, Any, Tuple

# --- Data Handling ---
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

# --- Machine Learning Metrics ---
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# --- Progress Bar ---
from tqdm import tqdm

# --- OpenAI Client ---
import openai

# Set global plot style
sns.set_style("whitegrid")


### **Data Loading**
### Loading and Understanding the Data


In [3]:
### Loading and Understanding the Data
# Load the dataset from 'data/Dataset - Real-Time Retail Feedback Intelligence.csv'. 
# Note: the separator is a semicolon (';') and the first column should be used as the index.

# Path to the dataset file:
data_path = "data/Dataset - Real-Time Retail Feedback Intelligence.csv"

# Read the CSV into a DataFrame:
# - sep=";"      -> the file uses a semicolon as the column separator
# - index_col=0  -> use the first (unnamed) column as the DataFrame index
df = pd.read_csv(data_path, sep=";", index_col=0)

# Display the loaded DataFrame to verify it was read correctly
display(df)

,Clothing.ID,Age,Title,Review.Text,Rating,Recommended.IND,Positive.Feedback.Count,Division.Name,Department.Name,Class.Name
1,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
2,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
3,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
4,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
5,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
...,...,...,...,...,...,...,...,...,...,...
23482,1104,34,Great dress for many occasions,I was very happy to snag this dress at such a ...,5,1,0,General Petite,Dresses,Dresses
23483,862,48,Wish it was made of cotton,"It reminds me of maternity clothes. soft, stre...",3,1,0,General Petite,Tops,Knits
23484,1104,31,"Cute, but see through","This fit well, but the top was very see throug...",3,0,1,General Petite,Dresses,Dresses
23485,1084,28,"Very cute dress, perfect for summer parties an...",I bought this dress for a wedding i have this ...,3,1,2,General,Dresses,Dresses


### **Data Overview**


#### Dimensions and First few rows of the dataset

In [13]:
# ----------------------------------------------------------------------
# Data Overview
# A quick, structured look at the dataset before any cleaning/analysis.
# ----------------------------------------------------------------------

# First few rows of the dataset
# head() shows the top 5 rows so we can eyeball the structure and values.
print("First few rows of the dataset:")
display(df.head())

# Dimensions
# shape returns a (rows, columns) tuple describing the size of the data.
print(f"\nDimensions (rows, columns): {df.shape}")
print(f"- Number of rows (records): {df.shape[0]}")
print(f"- Number of columns (features): {df.shape[1]}")



First few rows of the dataset:


,Clothing.ID,Age,Title,Review.Text,Rating,Recommended.IND,Positive.Feedback.Count,Division.Name,Department.Name,Class.Name
1,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
2,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
3,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
4,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
5,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses



Dimensions (rows, columns): (23486, 10)
- Number of rows (records): 23486
- Number of columns (features): 10


#### Column Details

In [14]:
# Column Names
# The list of all column labels in the DataFrame.
print("\nColumn Names:")
print(df.columns.tolist())

# Column Types, total and non-null Counts
# info() summarizes the dtype and the number of non-null values per column, which helps quickly spot columns with missing data.
print("\nColumn Types, total and non-null counts:")
df.info()



Column Names:
['Clothing.ID', 'Age', 'Title', 'Review.Text', 'Rating', 'Recommended.IND', 'Positive.Feedback.Count', 'Division.Name', 'Department.Name', 'Class.Name']

Column Types, total and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 23486 entries, 1 to 23486
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Clothing.ID              23486 non-null  int64
 1   Age                      23486 non-null  int64
 2   Title                    19676 non-null  str  
 3   Review.Text              22641 non-null  str  
 4   Rating                   23486 non-null  int64
 5   Recommended.IND          23486 non-null  int64
 6   Positive.Feedback.Count  23486 non-null  int64
 7   Division.Name            23472 non-null  str  
 8   Department.Name          23472 non-null  str  
 9   Class.Name               23472 non-null  str  
dtypes: int64(5), str(5)
memory usage: 1.8 MB


#### Statistical summary of numerical columns

In [15]:
# Statistical summary of numerical columns
# describe() reports count, mean, std, min, quartiles and max for numeric columns.
print("\nStatistical summary of numerical columns:")
display(df.describe())



Statistical summary of numerical columns:


,Clothing.ID,Age,Rating,Recommended.IND,Positive.Feedback.Count
count,23486.000000,23486.000000,23486.000000,23486.000000,23486.000000
mean,918.118709,43.198544,4.196032,0.822362,2.535936
std,203.298980,12.279544,1.110031,0.382216,5.702202
min,0.000000,18.000000,1.000000,0.000000,0.000000
25%,861.000000,34.000000,4.000000,1.000000,0.000000
50%,936.000000,41.000000,5.000000,1.000000,1.000000
75%,1078.000000,52.000000,5.000000,1.000000,3.000000
max,1205.000000,99.000000,5.000000,1.000000,122.000000


In [ ]:
# Memory Usage
# memory_usage(deep=True) accounts for the true size of object/string columns. Printing both per-column breakdown and the total footprint in MB.
print("\nMemory Usage (bytes per column):")
mem_usage = df.memory_usage(deep=True)
display(mem_usage)
print(f"- Total memory usage: {mem_usage.sum() / (1024 ** 2):.2f} MB")


Memory Usage (bytes per column):


Index                          132
Clothing.ID                 187888
Age                         187888
Title                      1461792
Review.Text                8127461
Rating                      187888
Recommended.IND             187888
Positive.Feedback.Count     187888
Division.Name              1374724
Department.Name            1284973
Class.Name                 1302280
dtype: int64

- Total memory usage: 13.82 MB


### **Sanity checks**

#### Duplicate Rows

In [16]:
# ----------------------------------------------------------------------
# Dataset Sanity Checks
# Basic data-quality checks before cleaning and analysis.
# ----------------------------------------------------------------------

# Duplicate Rows
# duplicated() identifies rows that exactly match an earlier row.
print("Duplicate Rows:")
num_duplicates = df.duplicated().sum()
print(f"- Total duplicate rows: {num_duplicates}")

# If duplicates exist, show a small sample to inspect them.
if num_duplicates > 0:
    print("- Sample duplicate rows:")
    display(df[df.duplicated(keep=False)].head())


Duplicate Rows:
- Total duplicate rows: 21
- Sample duplicate rows:


,Clothing.ID,Age,Title,Review.Text,Rating,Recommended.IND,Positive.Feedback.Count,Division.Name,Department.Name,Class.Name
299,1104,39,NaN,NaN,5,1,0,General,Dresses,Dresses
494,1104,39,NaN,NaN,5,1,0,General,Dresses,Dresses
1005,1094,30,NaN,NaN,5,1,0,General,Dresses,Dresses
2740,1094,36,NaN,NaN,5,1,0,General Petite,Dresses,Dresses
2942,829,66,NaN,NaN,5,1,0,General Petite,Tops,Blouses


#### Missing Values

In [18]:
# Missing Values
# isna() counts NaN/None values per column; percentages help compare columns.
print("\nMissing Values:")
missing_counts = df.isna().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percentage": missing_pct,
})

# Show only columns that actually have missing values.
display(missing_summary[missing_summary["missing_count"] > 0])



Missing Values:


,missing_count,missing_percentage
Title,3810,16.22
Review.Text,845,3.60
Division.Name,14,0.06
Department.Name,14,0.06
Class.Name,14,0.06


#### Unique Values

In [19]:

# Unique Values
# nunique() counts distinct values in each column (NaN is excluded by default).
print("\nUnique Values (per column):")
unique_counts = df.nunique()
display(unique_counts.to_frame(name="unique_count"))

# Non-empty and unique value counts for each column
# For text columns, blank strings are treated as empty in addition to NaN.
print("\nNon-empty and unique value counts for each column:")

def count_non_empty(series: pd.Series) -> int:
    """Count values that are present and not blank (for string/object columns)."""
    if series.dtype == "object" or pd.api.types.is_string_dtype(series):
        return series.dropna().astype(str).str.strip().ne("").sum()
    return series.notna().sum()

sanity_summary = pd.DataFrame({
    "non_empty_count": [count_non_empty(df[col]) for col in df.columns],
    "unique_count": [df[col].nunique() for col in df.columns],
}, index=df.columns)

# Derived count: rows that are missing or blank for each column.
sanity_summary["empty_or_missing_count"] = len(df) - sanity_summary["non_empty_count"]

display(sanity_summary)


Unique Values (per column):


,unique_count
Clothing.ID,1206
Age,77
Title,13993
Review.Text,22634
Rating,5
Recommended.IND,2
Positive.Feedback.Count,82
Division.Name,3
Department.Name,6
Class.Name,20



Non-empty and unique value counts for each column:


,non_empty_count,unique_count,empty_or_missing_count
Clothing.ID,23486,1206,0
Age,23486,77,0
Title,19676,13993,3810
Review.Text,22641,22634,845
Rating,23486,5,0
Recommended.IND,23486,2,0
Positive.Feedback.Count,23486,82,0
Division.Name,23472,3,14
Department.Name,23472,6,14
Class.Name,23472,20,14


### Data Observation Summary

In [20]:
# ----------------------------------------------------------------------
# Summary of all observations from Data Overview and Sanity Checks
# Consolidates key findings into a readable narrative before cleaning/EDA.
# ----------------------------------------------------------------------

# Recompute helper metrics so this cell can run independently if needed.
def count_non_empty(series: pd.Series) -> int:
    """Count values that are present and not blank (for string/object columns)."""
    if series.dtype == "object" or pd.api.types.is_string_dtype(series):
        return series.dropna().astype(str).str.strip().ne("").sum()
    return series.notna().sum()

column_counts = pd.DataFrame({
    "non_empty_count": [count_non_empty(df[col]) for col in df.columns],
    "unique_count": [df[col].nunique() for col in df.columns],
}, index=df.columns)

num_duplicates = df.duplicated().sum()
total_missing = df.isna().sum().sum()
missing_by_col = df.isna().sum()
cols_with_missing = missing_by_col[missing_by_col > 0]
int_cols = df.select_dtypes(include=["int64"]).columns.tolist()
str_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
total_memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print("=" * 80)
print("SUMMARY OF OBSERVATIONS")
print("=" * 80)

print("")
print("1. DATASET LOADING")
print(f"   - The Real-Time Retail Feedback Intelligence dataset has been loaded successfully.")
print(f"   - The dataset contains {df.shape[0]:,} rows and {df.shape[1]} columns.")
print(f"   - The file was read as a semicolon-delimited CSV with the first column used as the index.")
print(f"   - The first few rows show customer review records with product, rating, and category fields.")
print(f"   - No obvious loading issues such as misaligned columns or unexpected headers were observed.")

print("")
print("2. DATASET DIMENSIONS & COLUMNS")
print(f"   - Shape: {df.shape[0]:,} observations x {df.shape[1]} variables")
print(f"   - Columns: {', '.join(df.columns.tolist())}")
print(f"   - `Review.Text` is the most critical feature for LLM-based analysis.")
print(f"   - `Recommended.IND` is the binary recommendation label (1 = yes, 0 = no).")
print(f"   - `Rating`, `Division.Name`, `Department.Name`, and `Class.Name` provide review score and product context.")

print("")
print("3. DATA TYPES")
print(f"   - The dataset contains a mix of numeric and text fields suitable for NLP and classification tasks.")
print(f"   - {len(int_cols)} columns are int64: {', '.join(int_cols)}.")
print(f"   - {len(str_cols)} columns are string/object: {', '.join(str_cols)}.")
print(f"   - `Recommended.IND` is a binary indicator with values {sorted(df['Recommended.IND'].unique().tolist())}.")
print(f"   - `Rating` is an ordinal score from {df['Rating'].min()} to {df['Rating'].max()}.")

print("")
print("4. MISSING VALUES")
print(f"   - Total missing values in the dataset: {total_missing:,}")
if len(cols_with_missing) > 0:
    for col_name, miss_count in cols_with_missing.items():
        miss_pct = (miss_count / len(df) * 100)
        print(f"   - {col_name}: {miss_count:,} missing ({miss_pct:.2f}%)")
    print(f"   - Rows with missing `Review.Text` cannot be analyzed by the language model and should be dropped.")
    print(f"   - Rows with missing category fields can be dropped or imputed as 'Unknown'.")
else:
    print(f"   - Each column has 0 missing values.")
    print(f"   - No missing value treatment is required at this stage.")

print("")
print("5. DUPLICATE ROWS")
print(f"   - Number of duplicate rows: {num_duplicates}")
if num_duplicates > 0:
    print(f"   - A small number of exact duplicate records were found and should be reviewed during cleaning.")
else:
    print(f"   - The dataset does not contain duplicate records.")

print("")
print("6. NON-EMPTY & UNIQUE VALUE COUNTS")
print("   - Display counts in aligned columns for easier reading:")

# Display counts in aligned columns for easier reading.
aligned_counts = column_counts.astype(int).rename(
    columns={"non_empty_count": "Non-Empty", "unique_count": "Unique"}
)
aligned_counts.index.name = "Column"
print(aligned_counts.to_string())

print("")
print("7. KEY STATISTICAL OBSERVATIONS")
print(f"   - Rating: mean = {df['Rating'].mean():.2f}, median = {df['Rating'].median():.0f}, "
      f"std = {df['Rating'].std():.2f}, range = {df['Rating'].min()} to {df['Rating'].max()}")
print(f"   - Recommended.IND: mean = {df['Recommended.IND'].mean():.2f} "
      f"({df['Recommended.IND'].mean() * 100:.1f}% of reviews recommend the product)")
print(f"   - Age: mean = {df['Age'].mean():.2f}, range = {df['Age'].min()} to {df['Age'].max()}")
print(f"   - Positive.Feedback.Count: mean = {df['Positive.Feedback.Count'].mean():.2f}, "
      f"max = {df['Positive.Feedback.Count'].max()} (right-skewed; most reviews have few helpful votes)")
print(f"   - Clothing.ID: {df['Clothing.ID'].nunique():,} unique product IDs across the review records")
print(f"   - Division.Name: {df['Division.Name'].nunique()} unique values "
      f"({', '.join(df['Division.Name'].dropna().unique().astype(str).tolist())})")
print(f"   - Department.Name: {df['Department.Name'].nunique()} unique departments; "
      f"top department is '{df['Department.Name'].value_counts().idxmax()}'")
print(f"   - Class.Name: {df['Class.Name'].nunique()} unique product classes")

print("")
print("8. MEMORY USAGE")
print(f"   - Total memory usage: {total_memory_mb:.2f} MB")
print(f"   - Text columns (`Title`, `Review.Text`) contribute most of the memory footprint.")

print("")
print("9. OVERALL CONCLUSION")
print(f"   - The dataset is structurally consistent and suitable for retail feedback analysis.")
print(f"   - Missing review text and a small number of duplicate rows require attention before modeling.")
print(f"   - Numeric fields (`Rating`, `Recommended.IND`, `Positive.Feedback.Count`) are complete and valid.")
print(f"   - The data is ready for cleaning, exploratory data analysis, and LLM prompt experimentation.")

print("")
print("=" * 80)

SUMMARY OF OBSERVATIONS

1. DATASET LOADING
   - The Real-Time Retail Feedback Intelligence dataset has been loaded successfully.
   - The dataset contains 23,486 rows and 10 columns.
   - The file was read as a semicolon-delimited CSV with the first column used as the index.
   - The first few rows show customer review records with product, rating, and category fields.
   - No obvious loading issues such as misaligned columns or unexpected headers were observed.

2. DATASET DIMENSIONS & COLUMNS
   - Shape: 23,486 observations x 10 variables
   - Columns: Clothing.ID, Age, Title, Review.Text, Rating, Recommended.IND, Positive.Feedback.Count, Division.Name, Department.Name, Class.Name
   - `Review.Text` is the most critical feature for LLM-based analysis.
   - `Recommended.IND` is the binary recommendation label (1 = yes, 0 = no).
   - `Rating`, `Division.Name`, `Department.Name`, and `Class.Name` provide review score and product context.

3. DATA TYPES
   - The dataset contains a mix o

### **Data Cleaning and Preprocessing**

**Think about it:** The Review Text column is the most critical feature for our Generative AI model. What should be done with rows where this text is missing?

In [21]:
# ----------------------------------------------------------------------
# Data Cleaning and Preprocessing - Initial Cleaning (Before EDA)
# Goal: fix structural issues and remove unusable records only.
# Deeper missing-value treatment and text normalization will be performed after EDA.
# ----------------------------------------------------------------------

# Keep a snapshot of the original shape for a before/after summary.
rows_before = df.shape[0]
print("=" * 80)
print("INITIAL DATA CLEANING (BEFORE EDA)")
print("=" * 80)
print(f"Starting shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Work on a copy so the raw-loaded data is not overwritten in memory.
df_clean = df.copy()

# ------------------------------------------------------------------
# 1. Identify and remove duplicate rows
# ------------------------------------------------------------------
duplicate_count = df_clean.duplicated().sum()
print(f"\n1. Duplicate Rows")
print(f"   - Duplicate rows found: {duplicate_count}")

df_clean = df_clean.drop_duplicates(keep="first")
print(f"   - Rows removed: {duplicate_count}")
print(f"   - Shape after duplicate removal: {df_clean.shape[0]:,} rows")

# ------------------------------------------------------------------
# 2. Fix incorrect or mixed data types
# ------------------------------------------------------------------
print(f"\n2. Data Type Corrections")

# Numeric fields should be stored as integers for downstream analysis.
numeric_cols = [
    "Clothing.ID",
    "Age",
    "Rating",
    "Recommended.IND",
    "Positive.Feedback.Count",
]

for col in numeric_cols:
    # errors='coerce' turns any non-numeric values into NaN so they can be flagged later.
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Text fields should be consistently stored as pandas string dtype.
text_cols = ["Title", "Review.Text", "Division.Name", "Department.Name", "Class.Name"]
for col in text_cols:
    df_clean[col] = df_clean[col].astype("string")

# Cast valid numeric columns back to nullable integer types where appropriate.
for col in numeric_cols:
    df_clean[col] = df_clean[col].astype("Int64")

print(f"   - Enforced integer types for: {', '.join(numeric_cols)}")
print(f"   - Enforced string types for: {', '.join(text_cols)}")

# ------------------------------------------------------------------
# 3. Handle obvious, fatal errors in key numeric fields
# ------------------------------------------------------------------
print(f"\n3. Fatal Error Handling")

# Build a single mask for rows that are clearly invalid and cannot be used safely.
invalid_mask = (
    df_clean["Clothing.ID"].isna() | (df_clean["Clothing.ID"] <= 0)
    | df_clean["Age"].isna() | (df_clean["Age"] < 10) | (df_clean["Age"] > 100)
    | df_clean["Rating"].isna() | ~df_clean["Rating"].isin([1, 2, 3, 4, 5])
    | df_clean["Recommended.IND"].isna() | ~df_clean["Recommended.IND"].isin([0, 1])
    | df_clean["Positive.Feedback.Count"].isna() | (df_clean["Positive.Feedback.Count"] < 0)
)

fatal_error_count = invalid_mask.sum()
print(f"   - Rows with invalid numeric values: {fatal_error_count}")

# Review text is required for LLM-based analysis; blank or whitespace-only text is unusable.
blank_review_mask = df_clean["Review.Text"].fillna("").str.strip().eq("")
blank_review_count = blank_review_mask.sum()
print(f"   - Rows with missing or blank Review.Text: {blank_review_count}")

# Remove all fatal-error rows in one step.
rows_to_drop = invalid_mask | blank_review_mask
df_clean = df_clean.loc[~rows_to_drop].copy()
print(f"   - Total rows removed for fatal errors: {rows_to_drop.sum()}")
print(f"   - Shape after fatal-error removal: {df_clean.shape[0]:,} rows")

# ------------------------------------------------------------------
# 4. Other critical pre-EDA fixes (without deeper imputation yet)
# ------------------------------------------------------------------
print(f"\n4. Other Critical Pre-EDA Fixes")

# Strip leading/trailing whitespace from text columns to avoid hidden formatting issues.
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

# Replace empty strings created by stripping with missing values for consistency.
for col in text_cols:
    df_clean[col] = df_clean[col].replace("", pd.NA)

# Reset the index after row removals so downstream sampling/indexing is clean.
df_clean = df_clean.reset_index(drop=True)

print(f"   - Trimmed whitespace in text columns: {', '.join(text_cols)}")
print(f"   - Converted empty strings to missing values (no imputation yet)")
print(f"   - Reset index after row removals")

# Replace the working DataFrame used in later notebook steps.
df = df_clean

# ------------------------------------------------------------------
# Cleaning summary
# ------------------------------------------------------------------
rows_removed = rows_before - df.shape[0]
print(f"\n5. Cleaning Summary")
print(f"   - Rows before cleaning: {rows_before:,}")
print(f"   - Rows after cleaning:  {df.shape[0]:,}")
print(f"   - Total rows removed:   {rows_removed:,}")
print(f"   - Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

print(f"\nRemaining missing values (to be handled after EDA):")
display(df.isna().sum().to_frame(name="missing_count"))

print(f"\nFinal data types:")
df.info()

print("=" * 80)

SyntaxError: invalid syntax (1808370932.py, line 125)

### **Exploratory Data Analysis**

EDA is an important part of any project involving data. It is important to investigate and understand the data better before building a model with it. A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data. A thorough analysis of the data, in addition to the questions mentioned below, should be done.

**Questions:**

1.  What is the summary statistics of the numerical data? What can you infer about the distribution of Age, Rating, and Positive Feedback Count?
    
2.  How many unique values are there in the categorical columns like Division Name, Department Name, and Class Name?
    
3.  What is the overall distribution of product Rating? Is the dataset skewed towards positive or negative reviews?
    
4.  Which Department Name receives the highest average rating, and which receives the lowest? What might this indicate?
    
5.  What are the most common words found in highly-rated reviews (4-5 stars) versus poorly-rated reviews (1-2 stars)? (Hint: Use Word Clouds). What initial hypotheses can you form about the key drivers of customer satisfaction and dissatisfaction?

Also write your observations for each questions.

## **Building the Generative AI Pipeline**

We will now build a system to analyze the reviews. This involves setting up the AI client, designing prompts, generating structured data, and evaluating the results.

#### **Setup AI Client and Data Sample**

**Questions:**

1.  How do you initialize the OpenAI client with your API key and the correct base URL?
    

#### **Note:**

For this project, we will analyze and categorize a sample of **50 customer reviews**. This number is chosen intentionally. Since the API has a **budget limit of $20**, running prompts on very large datasets can quickly exhaust your quota—especially because this exercise may involve **multiple iterations, prompt refinements, and repeated evaluations**.

To avoid unnecessary cost and ensure efficient experimentation, we recommend the following approach:

*   **Use very small samples (5–10 reviews)** during the **initial testing phase** to validate your prompt structure and logic.
    
*   **Scale up to 50 reviews** for the **final evaluation phase**, ensuring you get enough data to compare prompting techniques without draining your budget.
    
*   This strategy helps maintain cost control while still providing meaningful insights across Zero-Shot, Few-Shot, and Chain-of-Thought techniques.
    

If your API quota gets exhausted, you may temporarily switch to another free AI assistant API. However, note that external tools may also have **rate limits** or **token caps**, so you will need to build retry logic and manage throttling within your code.

#### **Prompt Engineering and Evaluation**

We will test three different prompting techniques. For each, we will create a basic version (V1) and an enhanced version (V2).

**Think about it:** Why is it important to have a consistent and robust evaluation framework? How can we use an "LLM-as-Judge" to score the quality of our generated outputs objectively?

#### **Technique 1: Zero-Shot Prompting**

**Questions:**

1.  How would you design a basic Zero-Shot prompt that asks the model for Category, Sentiment, Summary, Personalized Message, and Retail Insight?
    
2.  How can you enhance this prompt with more business context (e.g., a company name, the importance of accuracy) to create a V2 prompt?
    
3.  How will you loop through the data sample to generate and store the structured output for both prompt versions?
    
4.  How will you apply the LLM-as-Judge to generate a evaluation score between 0 to 1 (decimal allowed) for the outputs and calculate the average score of V1 and V2 prompt?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your **Zero-Shot Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

#### **Technique 2: Few-Shot Prompting**

**Questions:**

1.  How do you structure a Few-Shot prompt? What kind of examples (e.g., one positive, one negative) would be most effective?
    
2.  For the V2 prompt, how can you add a set of "rules" to guide the model's output for each field, reducing ambiguity?
    
3.  After generating and scoring the outputs, how does the performance of Few-Shot prompting compare to previous version?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your ** Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

#### **Technique 3: Chain-of-Thought (CoT) Prompting**

**Questions:**

1.  How do you instruct the model to "think step-by-step" internally but only show the final, structured answer?
    
2.  How can you combine the CoT instruction with more detailed reasoning from the COT V1 prompt to create a powerful CoT V2 prompt?
    
3.  Does encouraging the model to reason first lead to a measurable improvement in the quality of the generated insights?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your **Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

## **Applying GenAI for Product Recommendation:**

Now, let's use the model for a different task: predicting the Recommended IND flag.

**Questions:**

1.  How do you design a prompt that strictly asks for a binary output (1 or 0) and a brief reason?
    
2.  What kind of function is needed to reliably parse the model's text response to extract the 1/0 flag and the Reason?
    
3.  How do you evaluate the model's performance as a classifier using standard metrics like accuracy, confusion matrix, and classification report?

**How the Process Works**


**1\. Prepare Data**

Copy the dataset, store the original recommendation labels, and remove them from the model input to avoid leakage.

**2\. Generate Predictions**

Use a strict two-line prompt to make the LLM output a binary recommendation (1/0) and a short reason based only on the review text.

**3\. Parse Outputs**

Extract the flag and reason from the raw LLM response using regex-based parsing that handles formatting issues.

**4\. Build Prediction Table**

Run the prompt for each review, parse the result, and store the predictions in a new DataFrame.

 **5\. Evaluate Performance**

Compare LLM predictions with true labels using accuracy, confusion matrix, and classification report.

 **6\. Explain Mismatches**

For incorrect predictions, generate a short explanation describing why the model’s decision may have differed from the human label.

**Visualization of Sentiments Distribution**

 After generating results from all prompting techniques, it's crucial to visualize their outputs to better understand their behavior and performance. This helps us see if one technique tends to be more cautious (e.g., assigning more 'Neutral' sentiments) or if they generally agree on the sentiment of the reviews.
    
 **Questions:**
    
* How does the distribution of predicted Sentiment (Positive, Negative, Neutral) compare across the V2 versions of Zero-Shot, Few-Shot, and Chain-of-Thought? (Hint: Create a separate bar chart for each technique's V2 sentiment column).
    
* Are there noticeable differences in the counts? For example, does one technique identify more "Neutral" reviews than the others? What might this imply about its ability to handle nuance?
    


##  **Comparison of Prompting Techniques:**
    
   *   How do the three techniques (Zero-Shot, Few-Shot, CoT) compare in terms of their responses. Use LLM to give verdict?
        
  *   Which technique was the most reliable and consistent? Why do you think it performed the best?
        
   *   What model and prompt design would you propose for a production environment?
        


### **Observations and Insights**

 **Refined Insights:**
    
   *   What are the most meaningful and recurring insights from the customer reviews, as identified by your best-performing model?

# Generating Actionable Product Improvement Suggestions


 *   Based on the aggregated insights from your best model, what are 3 short-term (3-6 months) and 3 long-term (6-12 months) actionable business recommendations for the retail company?
        
 *   How does this automated GenAI pipeline solve the initial business problem and create value?

### **Observations and Insights**

## **Conclusion**